# 🏡 San Antonio Automated Apartment & Townhome Hunter (v4.3 — Link Inspector, Favorites & Multi-Week Memory Edition)
### New Features in v4.3:
1. **Single-Link / Address Sanity Check Inspector (`Step 7`):** Paste *any* Realtor.com, Zillow, or Redfin link (or street address) to see a complete Waterfall Score Breakdown, check whether it failed any Hard Cutoff Gate, and see which `CONFIG['weights']` to tweak to surface more places like it.
2. **Long-Term Blacklist (Address + Entire Apartment Complex Blocking) & Favorites Shortlist (`Step 6`):** Block individual units or entire complexes (`blacklist_complex_name`), and maintain a persistent `outputs/favorites_shortlist.csv` tour sheet.
3. **Multi-Week Delta & Price-Drop Tracker (`seen_listings_history.json`):** Automatically flags `🆕 NEW`, `📉 PRICE DROP (-$X/mo)`, and `⭐ FAVORITE` across weekly runs.
4. **3-Tier Instant Caching (`Steps 2, 4, 5`):** Caches OpenStreetMap KD-Trees, raw HomeHarvest scrapes, and per-photo 512-d CLIP embeddings.

In [ ]:
# @title Step 0: Install Required Packages
!pip install -q homeharvest osmnx geopandas folium shapely scipy requests pandas pyyaml transformers torch pillow scikit-learn


In [ ]:
# @title Step 1: Configuration & Persistent Memory Loader (`preferences.json`)
import os
import re
import json
import math
import pickle
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
from shapely.geometry import Point
from scipy.spatial import cKDTree
from homeharvest import scrape_property

# Optional: Set USE_GOOGLE_DRIVE_MEMORY = True in Colab to persist cache & preferences across Colab resets!
USE_GOOGLE_DRIVE_MEMORY = False
PREF_FILE = 'data/preferences.json'
CACHE_DIR = 'cache'

if USE_GOOGLE_DRIVE_MEMORY:
    from google.colab import drive
    drive.mount('/content/drive')
    PREF_FILE = '/content/drive/MyDrive/LoveWhereYouLive_preferences.json'
    CACHE_DIR = '/content/drive/MyDrive/LoveWhereYouLive_cache'

os.makedirs(CACHE_DIR, exist_ok=True)

CONFIG = {
    'workplace': {'name': 'Southwest Research Institute (SwRI)', 'coords': (29.4485, -98.6105)},
    'costco_locations': [
        {'name': 'Costco NW (5611 UTSA Blvd)', 'coords': (29.5762, -98.6001)},
        {'name': 'Costco West (191 W Loop 1604 S)', 'coords': (29.4316, -98.7115)},
        {'name': 'Costco North Central (1201 N Loop 1604 E)', 'coords': (29.6095, -98.4716)},
        {'name': 'Costco NE Selma (15330 IH-35 N)', 'coords': (29.5794, -98.3188)}
    ],
    'filters': {'location': 'San Antonio, TX', 'min_rent': 1200, 'max_rent': 2200, 'min_beds': 2, 'min_baths': 1.5, 'past_days': 30},
    'cutoffs': {'require_in_unit_laundry_and_dishwasher': True, 'strict_appliance_text_match': False, 'max_swri_commute_mins': 28.0, 'max_grocery_dist_m': 2400.0, 'max_park_trail_dist_m': 1600.0},
    'weights': {
        'commute_score': 0.16,
        'grocery_score': 0.11,
        'park_trail_score': 0.11,
        'pollution_highway_score': 0.07,
        'modern_interior_score': 0.10,
        'garage_bike_score': 0.07,
        'costco_score': 0.07,
        'water_softener_score': 0.05,
        'pet_flooring_score': 0.05,
        'effective_rent_score': 0.05,
        'noise_buffer_score': 0.05,
        'recency_score': 0.06
    }
}

import os
import json
import datetime
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

PREF_PATH = "data/preferences.json"
HISTORY_PATH = "data/seen_listings_history.json"

def load_preferences(path=PREF_PATH):
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)
        default_prefs = {
            "version": "4.3",
            "taste_profile": {
                "liked_kitchen_photo_urls": [],
                "disliked_kitchen_photo_urls": [],
                "liked_property_addresses": [],
                "favorites_notes": {},
                "rejected_property_addresses": [],
                "blacklisted_complex_keywords": [],
                "rejection_reasons_log": [],
                "excluded_grocery_chains": ["walmart", "dollar general", "family dollar", "target"],
                "dealbreaker_keywords": ["shared laundry", "coin-operated", "no washer", "student housing", "room for rent"]
            },
            "learned_weights_override": {}
        }
        with open(path, "w") as f:
            json.dump(default_prefs, f, indent=2)
        return default_prefs
    with open(path, "r") as f:
        prefs = json.load(f)
    # Ensure backwards-compatible keys exist
    tp = prefs.setdefault("taste_profile", {})
    tp.setdefault("favorites_notes", {})
    tp.setdefault("blacklisted_complex_keywords", [])
    return prefs

def save_preferences(prefs, path=PREF_PATH):
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else ".", exist_ok=True)
    with open(path, "w") as f:
        json.dump(prefs, f, indent=2)

def record_feedback(
    address=None,
    liked=False,
    kitchen_photo_url=None,
    reason="",
    blacklist_complex_name=None,
    new_dealbreaker_keyword=None,
    path=PREF_PATH
):
    """
    Updates your Persistent Favorites Shortlist, Long-Term Property/Complex Blacklist,
    and CLIP Visual Taste Anchors in `preferences.json`.
    """
    prefs = load_preferences(path)
    tp = prefs["taste_profile"]
    today = datetime.date.today().isoformat()

    if address:
        if liked:
            if address not in tp["liked_property_addresses"]:
                tp["liked_property_addresses"].append(address)
            if address in tp["rejected_property_addresses"]:
                tp["rejected_property_addresses"].remove(address)
            if reason:
                tp["favorites_notes"][address] = f"[{today}] {reason}"
        else:
            if address not in tp["rejected_property_addresses"]:
                tp["rejected_property_addresses"].append(address)
            if address in tp["liked_property_addresses"]:
                tp["liked_property_addresses"].remove(address)

    if blacklist_complex_name:
        cname = blacklist_complex_name.lower().strip()
        if cname and cname not in tp["blacklisted_complex_keywords"]:
            tp["blacklisted_complex_keywords"].append(cname)

    if kitchen_photo_url:
        if liked and kitchen_photo_url not in tp["liked_kitchen_photo_urls"]:
            tp["liked_kitchen_photo_urls"].append(kitchen_photo_url)
        elif not liked and kitchen_photo_url not in tp["disliked_kitchen_photo_urls"]:
            tp["disliked_kitchen_photo_urls"].append(kitchen_photo_url)

    if new_dealbreaker_keyword:
        kw = new_dealbreaker_keyword.lower().strip()
        if kw and kw not in tp["dealbreaker_keywords"]:
            tp["dealbreaker_keywords"].append(kw)

    if reason:
        tp["rejection_reasons_log"].append({
            "date": today,
            "address": address,
            "liked": liked,
            "kitchen_photo_url": kitchen_photo_url,
            "reason": reason
        })

    save_preferences(prefs, path)
    print(f"✅ Memory Updated ({path})!")
    print(f"  ⭐ Favorites Shortlist: {len(tp['liked_property_addresses'])} properties")
    print(f"  🚫 Blacklist: {len(tp['rejected_property_addresses'])} addresses | {len(tp['blacklisted_complex_keywords'])} complex names")
    print(f"  🖼️ Kitchen Vision Anchors: {len(tp['liked_kitchen_photo_urls'])} Liked | {len(tp['disliked_kitchen_photo_urls'])} Disliked")
    return prefs

def track_multi_week_changes_and_favorites(scored_df, prefs, history_path=HISTORY_PATH, fav_csv_path="outputs/favorites_shortlist.csv"):
    """
    Multi-Week Delta Tracker:
    - Tags every listing in `scored_df` with `listing_delta_status`:
      '⭐ FAVORITE', '🆕 NEW LISTING', '📉 PRICE DROP (-$X)', or 'Seen Previously'.
    - Exports your Favorites Shortlist to `outputs/favorites_shortlist.csv` so you have a clean tour sheet!
    """
    os.makedirs(os.path.dirname(history_path) if os.path.dirname(history_path) else ".", exist_ok=True)
    history = {}
    if os.path.exists(history_path):
        try:
            with open(history_path, "r") as f:
                history = json.load(f)
        except Exception:
            history = {}

    today = datetime.date.today().isoformat()
    fav_set = set(prefs.get("taste_profile", {}).get("liked_property_addresses", []))
    fav_notes = prefs.get("taste_profile", {}).get("favorites_notes", {})

    statuses = []
    for _, row in scored_df.iterrows():
        addr = str(row.get("formatted_address", ""))
        price = float(row.get("list_price", 0))
        prefix = "⭐ FAVORITE | " if addr in fav_set else ""

        if addr not in history:
            statuses.append(prefix + "🆕 NEW")
            history[addr] = {"first_seen": today, "last_seen": today, "first_price": price, "last_price": price}
        else:
            prev_price = float(history[addr].get("last_price", price))
            history[addr]["last_seen"] = today
            if price < prev_price:
                drop = int(prev_price - price)
                statuses.append(prefix + f"📉 PRICE DROP (-${drop}/mo)")
                history[addr]["last_price"] = price
            elif price > prev_price:
                statuses.append(prefix + f"📈 Price Up (+${int(price - prev_price)}/mo)")
                history[addr]["last_price"] = price
            else:
                statuses.append(prefix + f"Seen ({history[addr].get('first_seen', today)})")

    scored_df = scored_df.copy()
    scored_df["listing_status"] = statuses
    scored_df["is_favorite"] = scored_df["formatted_address"].isin(fav_set)
    scored_df["favorite_notes"] = scored_df["formatted_address"].map(lambda a: fav_notes.get(a, ""))

    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    if fav_set:
        os.makedirs(os.path.dirname(fav_csv_path) if os.path.dirname(fav_csv_path) else ".", exist_ok=True)
        fav_df = scored_df[scored_df["is_favorite"]].copy()
        fav_df.to_csv(fav_csv_path, index=False)
        print(f"⭐ Exported {len(fav_df)} shortlisted favorites to {fav_csv_path}!")

    return scored_df

def calibrate_weights_from_ratings(scored_df, user_ratings_dict, base_weights, path=PREF_PATH):
    score_cols = list(base_weights.keys())
    X, y = [], []
    for addr_sub, target_score in user_ratings_dict.items():
        matches = scored_df[scored_df['formatted_address'].astype(str).str.contains(addr_sub, case=False, na=False)]
        if len(matches) > 0:
            row = matches.iloc[0]
            X.append([float(row.get(c, 50.0)) for c in score_cols])
            y.append(float(target_score))

    if len(X) < 3:
        print("Need at least 3 rated properties to auto-tune weights! Keeping current weights.")
        return base_weights

    model = Ridge(alpha=10.0, positive=True)
    model.fit(np.array(X), np.array(y))
    coefs = model.coef_
    if coefs.sum() > 0:
        learned = coefs / coefs.sum()
        new_weights = {}
        for col, lw in zip(score_cols, learned):
            new_weights[col] = round(float(0.50 * base_weights[col] + 0.50 * lw), 3)
        total = sum(new_weights.values())
        new_weights = {k: round(v / total, 3) for k, v in new_weights.items()}
        prefs = load_preferences(path)
        prefs["learned_weights_override"] = new_weights
        save_preferences(prefs, path)
        print("Learned new weights from your ratings:", new_weights)
        return new_weights
    return base_weights

PREFS = load_preferences(PREF_FILE)
if PREFS.get('learned_weights_override'):
    CONFIG['weights'].update(PREFS['learned_weights_override'])
print(f'✅ Loaded Configuration & Persistent Memory from {PREF_FILE}!')
print(f"Memory Stats -> ⭐ Favorites: {len(PREFS['taste_profile']['liked_property_addresses'])} | 🚫 Blacklisted: {len(PREFS['taste_profile']['rejected_property_addresses'])} | 🖼️ Kitchen Anchors: +{len(PREFS['taste_profile']['liked_kitchen_photo_urls'])} / -{len(PREFS['taste_profile']['disliked_kitchen_photo_urls'])}")


In [ ]:
# @title Step 2: Build or Load Cached Spatial KD-Trees (Groceries [No Walmart], Parks, Highways, Railways)
loc = CONFIG['filters']['location']
OSM_CACHE_FILE = os.path.join(CACHE_DIR, f"osm_trees_{loc.replace(', ', '_').replace(' ', '_').lower()}.pkl")
FORCE_REFRESH_OSM_CACHE = False  # Set to True only if you want to re-download fresh OSM data

def get_osm_tree(place, tags):
    gdf = ox.features_from_place(place, tags=tags).to_crs(epsg=32614)
    xy = np.array([(geom.centroid.x, geom.centroid.y) for geom in gdf.geometry])
    names = gdf['name'].fillna('Unnamed').tolist() if 'name' in gdf.columns else ['Unnamed'] * len(xy)
    return cKDTree(xy), names

if os.path.exists(OSM_CACHE_FILE) and not FORCE_REFRESH_OSM_CACHE:
    print(f"⚡ Loading cached spatial KD-Trees from {OSM_CACHE_FILE} (instant)...")
    with open(OSM_CACHE_FILE, "rb") as f:
        TREES = pickle.load(f)
    print(f"✅ Loaded {len(TREES)} spatial layers from cache in <0.1s!")
else:
    print(f"⏳ Fetching OpenStreetMap geospatial layers for {loc} (runs once, then caches)...")
    TREES = {
        'grocery': get_osm_tree(loc, {'shop': ['supermarket', 'grocery', 'greengrocer']}),
        'park': get_osm_tree(loc, {'leisure': ['park', 'nature_reserve'], 'highway': ['cycleway', 'path', 'footway']}),
        'highway': get_osm_tree(loc, {'highway': ['motorway', 'trunk']}),
        'railway': get_osm_tree(loc, {'railway': 'rail'})
    }
    with open(OSM_CACHE_FILE, "wb") as f:
        pickle.dump(TREES, f)
    print(f"✅ Saved spatial KD-Trees to persistent cache: {OSM_CACHE_FILE}")


In [ ]:
# @title Step 3: Register All 12 Modular Criteria Functions
import math
import re
import numpy as np
import pandas as pd

CIRCUITY_FACTOR = 1.32
WALK_SPEED_M_MIN = 80.0
BIKE_SPEED_M_MIN = 250.0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def criterion_required_appliances_gatekeeper(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    year_built = row.get('year_built', np.nan)
    strict = ctx['config']['cutoffs'].get('strict_appliance_text_match', False)

    shared_only_flags = ['shared laundry', 'laundromat on site', 'no washer', 'no laundry in unit', 'coin-operated laundry']
    if any(flag in desc for flag in shared_only_flags):
        return {'appliances_status': 'REJECT: Shared/No In-Unit Laundry', 'passes_appliance_gate': False}

    in_unit_terms = ['in-unit', 'in unit', 'washer/dryer', 'washer and dryer', 'w/d included', 'w/d connections', 'laundry room', 'utility room', 'full-size washer']
    has_in_unit = any(t in desc for t in in_unit_terms)
    has_dishwasher = 'dishwasher' in desc or ('stainless' in desc and 'appliances' in desc) or (pd.notna(year_built) and year_built >= 2005)

    if has_in_unit and has_dishwasher:
        return {'appliances_status': 'Verified (In-Unit W/D + Dishwasher)', 'passes_appliance_gate': True}
    elif strict:
        return {'appliances_status': 'REJECT: Unconfirmed in Text', 'passes_appliance_gate': False}
    return {'appliances_status': 'Likely (Verify W/D in Photos)', 'passes_appliance_gate': True}

def criterion_garage_and_bike_storage(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    prop_type = str(row.get('style', '') or row.get('type', '') or '').lower()
    parking_spaces = row.get('parking_garage', np.nan)

    score = 30.0
    features = []
    if pd.notna(parking_spaces) and float(parking_spaces) >= 1:
        score += 45.0
        features.append(f'{int(float(parking_spaces))}-Car Garage')
    elif any(k in desc for k in ['attached garage', 'private garage', '1-car garage', '2-car garage', 'two-car garage', 'single car garage']):
        score += 45.0
        features.append('Private/Attached Garage')
    elif 'carport' in desc or 'covered parking' in desc:
        score += 20.0
        features.append('Covered/Carport')

    if any(k in desc for k in ['bike storage', 'bicycle storage', 'bike rack', 'bike repair', 'storage unit', 'storage closet']):
        score += 25.0
        features.append('Dedicated Bike/Gear Storage')
    if 'townhome' in prop_type or 'townhouse' in prop_type or 'townhome' in desc or 'first floor' in desc:
        score += 15.0
        features.append('Townhome/Ground Access')

    return {
        'garage_bike_features': ', '.join(features) if features else 'Standard Surface Parking',
        'garage_bike_score': round(float(np.clip(score, 0.0, 100.0)), 1)
    }

def criterion_costco_proximity(row, ctx):
    lat, lon = row['latitude'], row['longitude']
    best_name, min_road_km = '', 999.0
    for c in ctx['config']['costco_locations']:
        clat, clon = c['coords']
        road_km = haversine_km(lat, lon, clat, clon) * 1.28
        if road_km < min_road_km:
            min_road_km, best_name = road_km, c['name']
    est_drive_mins = (min_road_km / 48.0) * 60.0 + 2.0
    score = np.clip(100 - ((est_drive_mins - 7.0) / (25.0 - 7.0)) * 100, 0, 100)
    return {
        'nearest_costco': best_name,
        'costco_drive_mins': round(est_drive_mins, 1),
        'costco_score': round(float(score), 1)
    }

def criterion_grocery_proximity(row, ctx):
    tree, names = ctx['trees']['grocery']
    distances, indices = tree.query([row['utm_x'], row['utm_y']], k=min(5, len(names)))
    
    if not isinstance(indices, np.ndarray):
        indices = [indices]
        distances = [distances]
        
    best_idx = indices[0]
    best_dist = distances[0]
    best_name = names[best_idx]
    
    for idx, dist in zip(indices, distances):
        name = names[idx].lower()
        if 'walmart' not in name:
            best_idx = idx
            best_dist = dist
            best_name = names[idx]
            break
            
    network_dist_m = best_dist * CIRCUITY_FACTOR
    walk_mins = network_dist_m / WALK_SPEED_M_MIN
    bike_mins = network_dist_m / BIKE_SPEED_M_MIN
    score = np.clip(100 - ((network_dist_m - 400) / (2400 - 400)) * 100, 0, 100)
    return {
        'nearest_grocery': best_name,
        'grocery_dist_m': round(network_dist_m),
        'grocery_walk_mins': round(walk_mins, 1),
        'grocery_bike_mins': round(bike_mins, 1),
        'grocery_score': round(float(score), 1)
    }

def criterion_park_trail_proximity(row, ctx):
    tree, names = ctx['trees']['park']
    dist_euclidean, idx = tree.query([row['utm_x'], row['utm_y']])
    network_dist_m = dist_euclidean * 1.25
    walk_mins = network_dist_m / WALK_SPEED_M_MIN
    score = np.clip(100 - ((network_dist_m - 250) / (1600 - 250)) * 100, 0, 100)
    return {
        'nearest_park_trail': names[idx],
        'park_dist_m': round(network_dist_m),
        'park_walk_mins': round(walk_mins, 1),
        'park_trail_score': round(float(score), 1)
    }

def criterion_swri_rush_hour_commute(row, ctx):
    lat, lon = row['latitude'], row['longitude']
    swri_lat, swri_lon = ctx['config']['workplace']['coords']
    road_km = haversine_km(lat, lon, swri_lat, swri_lon) * 1.30
    if lon < -98.68:
        avg_speed_kmh, corridor_note = 28.0, 'West Inbound (Heavy Culebra/1604 Traffic)'
    elif lon > -98.58:
        avg_speed_kmh, corridor_note = 50.0, 'Reverse Commute (Central/North)'
    else:
        avg_speed_kmh, corridor_note = 40.0, 'Inner West / Loop 410 Local'
    est_commute_mins = (road_km / avg_speed_kmh) * 60.0 + 3.0
    score = np.clip(100 - ((est_commute_mins - 8.0) / (30.0 - 8.0)) * 100, 0, 100)
    return {
        'swri_road_km': round(road_km, 1),
        'swri_rush_hour_mins': round(est_commute_mins, 1),
        'commute_corridor': corridor_note,
        'commute_score': round(float(score), 1)
    }

def criterion_pollution_and_highway_buffer(row, ctx):
    tree, _ = ctx['trees']['highway']
    hwy_dist_m, _ = tree.query([row['utm_x'], row['utm_y']])
    if hwy_dist_m < 100:
        score = 15.0
    elif hwy_dist_m < 250:
        score = 50.0 + ((hwy_dist_m - 100) / 150.0) * 35.0
    else:
        score = np.clip(85.0 + ((hwy_dist_m - 250) / 500.0) * 15.0, 85.0, 100.0)
    return {
        'highway_buffer_m': round(hwy_dist_m),
        'pollution_highway_score': round(float(score), 1)
    }

def criterion_modern_appliances_and_open_floorplan(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    year_built = row.get('year_built', np.nan)
    score, matched_tags = 50.0, []
    positive_keywords = {
        'open floor': 18, 'open-concept': 18, 'open concept': 18,
        'stainless steel': 15, 'stainless appliances': 15,
        'quartz': 12, 'granite': 10, 'kitchen island': 12, 'island kitchen': 12,
        'gas range': 12, 'gas stove': 12, 'modern': 8, 'renovated': 10
    }
    for kw, pts in positive_keywords.items():
        if kw in desc:
            score += pts
            matched_tags.append(kw)
    if pd.notna(year_built) and year_built >= 2018:
        score += 20
        matched_tags.append(f'built_{int(year_built)}')
    return {
        'interior_tags': ', '.join(matched_tags) if matched_tags else 'Standard/Unlisted',
        'modern_interior_score': round(float(np.clip(score, 10.0, 100.0)), 1)
    }

def criterion_water_softener_and_filtration(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    score, tags = 40.0, []
    softener_keywords = {
        'water softener': 40, 'softener loop': 35, 'softened water': 35,
        'reverse osmosis': 25, 'ro system': 25, 'filtration system': 20,
        'water filtration': 20, 'tankless water heater': 15
    }
    for kw, pts in softener_keywords.items():
        if kw in desc:
            score += pts
            tags.append(kw)
    return {
        'water_softener_tags': ', '.join(tags) if tags else 'Standard Hard Water (No Softener Listed)',
        'water_softener_score': round(float(np.clip(score, 10.0, 100.0)), 1)
    }

def criterion_pet_friendly_and_hard_flooring(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    score, tags = 60.0, []
    hard_flooring_terms = ['vinyl plank', 'lvp', 'wood floor', 'hardwood', 'tile flooring', 'stained concrete', 'no carpet']
    carpet_flags = ['wall-to-wall carpet', 'carpet in bedrooms', 'new carpet']
    cat_terms = ['cats allowed', 'cat friendly', 'pets allowed', 'pet friendly', 'no pet rent']
    
    if any(term in desc for term in hard_flooring_terms):
        score += 25.0
        tags.append('Hard Flooring / LVP')
    elif any(flag in desc for flag in carpet_flags):
        score -= 20.0
        tags.append('Carpeted Bedrooms')
    if any(term in desc for term in cat_terms):
        score += 15.0
        tags.append('Pet/Cat Friendly')
        
    return {
        'pet_flooring_tags': ', '.join(tags) if tags else 'Standard Flooring',
        'pet_flooring_score': round(float(np.clip(score, 10.0, 100.0)), 1)
    }

def criterion_advanced_junk_fee_parser(row, ctx):
    desc = str(row.get('description', '') or '').lower()
    fee_patterns = r'\$(\d+)\s*(?:/mo)?\s*(?:valet|trash|waste|package|internet|tech|pest|amenity|fee)'
    matches = re.findall(fee_patterns, desc)
    extracted_fees = sum([int(m) for m in matches])
    utilities_included = any(k in desc for k in ['water included', 'trash included', 'internet included', 'free wifi', 'all bills paid'])
    
    score = 100.0 - min(extracted_fees, 50.0)
    if utilities_included:
        score += 10.0
    return {
        'extracted_monthly_fees': extracted_fees,
        'utilities_included': utilities_included,
        'effective_rent_score': round(float(np.clip(score, 30.0, 100.0)), 1)
    }

def criterion_spatial_noise_buffer(row, ctx):
    lat, lon = row['latitude'], row['longitude']
    utm_x, utm_y = row['utm_x'], row['utm_y']
    
    rail_dist_m = 9999.0
    if 'railway' in ctx['trees']:
        rail_tree, _ = ctx['trees']['railway']
        rail_dist_m, _ = rail_tree.query([utm_x, utm_y])
        
    sat_coords = (29.5338, -98.4698)
    sat_dist_km = haversine_km(lat, lon, sat_coords[0], sat_coords[1])
    
    score, notes = 90.0, []
    if rail_dist_m < 200:
        score -= 40.0
        notes.append(f'Close to Railway ({round(rail_dist_m)}m)')
    elif rail_dist_m < 500:
        score -= 15.0
        notes.append(f'Moderate Rail Proximity ({round(rail_dist_m)}m)')
    if sat_dist_km < 6.0:
        score -= 25.0
        notes.append(f'SAT Airport Approach ({round(sat_dist_km, 1)}km)')
        
    return {
        'railway_dist_m': round(rail_dist_m) if rail_dist_m < 9000 else 9999,
        'sat_airport_dist_km': round(sat_dist_km, 1),
        'noise_buffer_notes': ', '.join(notes) if notes else 'Low Noise Corridor',
        'noise_buffer_score': round(float(np.clip(score, 20.0, 100.0)), 1)
    }


def criterion_year_built_recency(row, ctx):
    """CRITERION: Favors newer construction properties (year_built) to ensure modern structural standards."""
    year_built = row.get('year_built', np.nan)
    if pd.isna(year_built):
        return {'property_age_years': 'Unknown', 'recency_score': 50.0}
    current_year = 2026
    age_years = max(0, current_year - int(year_built))
    if age_years <= 3:
        score = 100.0
    elif age_years <= 7:
        score = 85.0
    elif age_years <= 12:
        score = 70.0
    elif age_years <= 20:
        score = 50.0
    else:
        score = np.clip(100.0 - (age_years * 2.5), 15.0, 45.0)
    return {'property_age_years': age_years, 'recency_score': round(float(score), 1)}

ACTIVE_CRITERIA = [
    criterion_year_built_recency,
    criterion_required_appliances_gatekeeper,
    criterion_garage_and_bike_storage,
    criterion_costco_proximity,
    criterion_grocery_proximity,
    criterion_park_trail_proximity,
    criterion_swri_rush_hour_commute,
    criterion_pollution_and_highway_buffer,
    criterion_modern_appliances_and_open_floorplan,
    criterion_water_softener_and_filtration,
    criterion_pet_friendly_and_hard_flooring,
    criterion_advanced_junk_fee_parser,
    criterion_spatial_noise_buffer
]

print(f'✅ Registered {len(ACTIVE_CRITERIA)} criteria functions!')


In [ ]:
# @title Step 4: Scrape (or Load Cached) Listings & Compute Spatial/Text Scores (`scored_df`)
# Caching the raw HomeHarvest scrape for 7 days (until you flip FORCE_REFRESH_SCRAPE = True) lets you tweak criteria/weights in Step 3 and re-run Step 4 in <1 second!
SCRAPE_CACHE_FILE = os.path.join(CACHE_DIR, "raw_listings_san_antonio.parquet")
FORCE_REFRESH_SCRAPE = False  # Set True to force a fresh live scrape from HomeHarvest
SCRAPE_TTL_HOURS = 168.0  # 7 days (or set FORCE_REFRESH_SCRAPE = True anytime)

use_cached_scrape = (
    os.path.exists(SCRAPE_CACHE_FILE)
    and not FORCE_REFRESH_SCRAPE
    and ((time.time() - os.path.getmtime(SCRAPE_CACHE_FILE)) / 3600.0 < SCRAPE_TTL_HOURS)
)

if use_cached_scrape:
    print(f"⚡ Loading cached rental listings from {SCRAPE_CACHE_FILE}...")
    raw = pd.read_parquet(SCRAPE_CACHE_FILE)
else:
    print("⏳ Scraping active rental listings in San Antonio via HomeHarvest...")
    raw = scrape_property(location=CONFIG['filters']['location'], listing_type='for_rent', past_days=CONFIG['filters']['past_days'])
    try:
        raw.to_parquet(SCRAPE_CACHE_FILE, index=False)
        print(f"✅ Cached {len(raw)} raw listings to {SCRAPE_CACHE_FILE} (valid for {SCRAPE_TTL_HOURS}h)")
    except Exception:
        pass

df = raw.dropna(subset=['latitude', 'longitude', 'list_price']).copy()
f = CONFIG['filters']
df = df[
    (df['list_price'] >= f['min_rent']) &
    (df['list_price'] <= f['max_rent']) &
    (df['beds'].fillna(0) >= f['min_beds']) &
    (df['full_baths'].fillna(0) >= 1)
].copy()

# Filter out any addresses or dealbreaker keywords stored in your Persistent Memory!
PREFS = load_preferences(PREF_FILE)
rejected_addrs = set(PREFS['taste_profile'].get('rejected_property_addresses', []))
dealbreakers = [kw.lower() for kw in PREFS['taste_profile'].get('dealbreaker_keywords', [])]
blacklisted_complexes = [kw.lower() for kw in PREFS['taste_profile'].get('blacklisted_complex_keywords', [])]

if rejected_addrs:
    before_len = len(df)
    df = df[~df['formatted_address'].isin(rejected_addrs)].copy()
    print(f'Memory Filter: Dropped {before_len - len(df)} previously rejected properties.')

gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs='EPSG:4326').to_crs(epsg=32614)
df['utm_x'], df['utm_y'] = gdf.geometry.x, gdf.geometry.y

ctx = {'config': CONFIG, 'trees': TREES}
results = []
weights = CONFIG['weights']

for _, row in df.iterrows():
    desc_lower = str(row.get('description', '') or '').lower()
    addr_lower = str(row.get('formatted_address', '') or '').lower()
    if any(db in desc_lower for db in dealbreakers) or any(bc in addr_lower or bc in desc_lower for bc in blacklisted_complexes):
        results.append({'passes_appliance_gate': False})
        continue
    m = {}
    for fn in ACTIVE_CRITERIA:
        m.update(fn(row, ctx))
    if m.get('passes_appliance_gate', True):
        m['composite_score'] = round(sum(m.get(k, 50.0) * w for k, w in weights.items()), 1)
    results.append(m)

full_scored = pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)
c_cfg = CONFIG['cutoffs']
scored_df = full_scored[
    (full_scored['passes_appliance_gate'] == True) &
    (full_scored['swri_rush_hour_mins'] <= c_cfg['max_swri_commute_mins']) &
    (full_scored['grocery_dist_m'] <= c_cfg['max_grocery_dist_m']) &
    (full_scored['park_dist_m'] <= c_cfg['max_park_trail_dist_m'])
].sort_values(by='composite_score', ascending=False).reset_index(drop=True)

print(f'✅ Step 4 Complete! `scored_df` created with {len(scored_df)} surviving candidates.')
scored_df[['composite_score', 'formatted_address', 'list_price', 'property_age_years', 'swri_rush_hour_mins', 'nearest_grocery', 'nearest_costco']].head(10)


In [ ]:
# @title Step 5: Run 5-Axis Kitchen Vision Grader (With Persistent Per-Photo Embedding Cache!) + Export Audit HTML
import io
import os
import pickle
import requests
from PIL import Image
import numpy as np
import pandas as pd

def score_photos_with_local_clip(
    df_top,
    max_candidates=25,
    liked_image_urls=None,
    disliked_image_urls=None,
    export_audit_html="outputs/visual_kitchen_audit.html",
    cache_path="cache/clip_photo_cache.pkl",
    force_refresh_vision=False
):
    """
    Robust 3-Stage Local Vision Grader (OpenAI CLIP) with Persistent Per-URL Disk Caching:
    - Caches room classification, 5-axis scores, and 512-d image embeddings per photo URL in `cache_path`.
    - If a photo URL was already downloaded & evaluated once, it never re-downloads or re-runs CLIP on it!
    - Even better: when you add new `liked_image_urls` or `disliked_image_urls` in Step 6, it computes
      new taste similarity scores in <0.05 seconds directly from the cached 512-d embeddings!
    """
    os.makedirs(os.path.dirname(cache_path) if os.path.dirname(cache_path) else ".", exist_ok=True)
    photo_cache = {}
    if os.path.exists(cache_path) and not force_refresh_vision:
        try:
            with open(cache_path, "rb") as f:
                photo_cache = pickle.load(f)
            print(f"⚡ Loaded CLIP vision cache with {len(photo_cache)} pre-analyzed photo URLs from {cache_path}")
        except Exception:
            photo_cache = {}

    # Lazy-load CLIP model ONLY if we encounter an uncached URL!
    model_state = {"model": None, "processor": None, "device": None}

    def ensure_clip_loaded():
        if model_state["model"] is None:
            try:
                from transformers import CLIPProcessor, CLIPModel
                import torch
            except ImportError:
                import subprocess
                subprocess.check_call(["pip", "install", "-q", "transformers", "torch", "pillow"])
                from transformers import CLIPProcessor, CLIPModel
                import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
            print(f"⏳ Loading Local CLIP Vision Engine on {device} for uncached photos...")
            model_state["device"] = device
            model_state["model"] = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
            model_state["processor"] = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    room_labels = [
        "an indoor kitchen with stove, refrigerator, sink, countertops, and cabinets", # 0: Kitchen
        "an empty indoor living room or bedroom showing floors and walls",             # 1: Living/Bedroom
        "an indoor bathroom with toilet, shower, or bathtub",                          # 2: Bathroom
        "an outdoor building exterior, swimming pool, patio, street, or grass",        # 3: Exterior
        "a 2D floor plan diagram, map, or text logo"                                   # 4: Diagram
    ]

    kitchen_axes = {
        "appliances": [
            "modern stainless steel kitchen appliances, sleek induction or gas range, stainless refrigerator",
            "old white or black plastic kitchen appliances, exposed electric coil burners, cheap stove"
        ],
        "counters": [
            "luxury thick quartz, marble, or polished granite stone kitchen countertops with undermount sink",
            "cheap laminate formica countertops, dated beige or speckled counter, or old drop-in sink"
        ],
        "cabinets": [
            "modern shaker style or sleek flat-panel contemporary kitchen cabinets with modern hardware",
            "dated 1980s or 1990s orange oak wood cabinets, cheap builder-grade brown wood doors, or worn cabinets"
        ],
        "layout": [
            "spacious open-concept kitchen with a large center island or breakfast bar overlooking the living room",
            "cramped narrow galley kitchen, closed-off small apartment kitchen with no island and tight walls"
        ]
    }

    flooring_axis = [
        "clean modern luxury vinyl plank flooring, wide-plank hardwood floors, or modern large format tile",
        "wall-to-wall beige carpet, plush bedroom carpeting, or dated linoleum squares"
    ]

    cache_dirty = [False]

    def analyze_or_get_cached_url(url):
        """Returns cached dict for a photo URL, or downloads & runs CLIP once and caches it."""
        if url in photo_cache and not force_refresh_vision:
            return photo_cache[url]

        ensure_clip_loaded()
        import torch
        model = model_state["model"]
        processor = model_state["processor"]
        device = model_state["device"]

        try:
            resp = requests.get(url, timeout=3.5)
            img = Image.open(io.BytesIO(resp.content)).convert("RGB")

            # 1. Image embedding (normalized 512-d vector)
            img_inputs = processor(images=img, return_tensors="pt").to(device)
            with torch.no_grad():
                feat = model.get_image_features(**img_inputs)
                feat = feat / feat.norm(p=2, dim=-1, keepdim=True)
            emb = feat.cpu().numpy()[0]

            # 2. Room classification
            r_inputs = processor(text=room_labels, images=img, return_tensors="pt", padding=True).to(device)
            with torch.no_grad():
                r_probs = model(**r_inputs).logits_per_image.softmax(dim=1).cpu().numpy()[0]
            k_prob, l_prob, b_prob, ext_prob, diag_prob = [float(x) for x in r_probs]

            entry = {
                "valid": True,
                "emb": emb,
                "k_prob": k_prob,
                "l_prob": l_prob,
                "b_prob": b_prob,
                "ext_prob": ext_prob,
                "diag_prob": diag_prob,
                "axes": {},
                "floor": 50.0
            }

            is_rejected_room = (ext_prob > 0.35 or diag_prob > 0.35 or b_prob > 0.40)
            if not is_rejected_room and (k_prob >= 0.38 or l_prob >= 0.38):
                # Compute flooring score
                f_inputs = processor(text=flooring_axis, images=img, return_tensors="pt", padding=True).to(device)
                with torch.no_grad():
                    f_prob = model(**f_inputs).logits_per_image.softmax(dim=1).cpu().numpy()[0][0]
                entry["floor"] = float(f_prob * 100.0)

                if k_prob >= 0.38:
                    for axis_name, pair_prompts in kitchen_axes.items():
                        ax_inputs = processor(text=pair_prompts, images=img, return_tensors="pt", padding=True).to(device)
                        with torch.no_grad():
                            ax_prob = model(**ax_inputs).logits_per_image.softmax(dim=1).cpu().numpy()[0][0]
                        entry["axes"][axis_name] = float(ax_prob * 100.0)

            photo_cache[url] = entry
            cache_dirty[0] = True
            return entry
        except Exception:
            entry = {"valid": False}
            photo_cache[url] = entry
            cache_dirty[0] = True
            return entry

    # Pre-fetch embeddings for user's Liked / Disliked reference kitchen URLs (also cached!)
    liked_embs, disliked_embs = [], []
    if liked_image_urls:
        for u in liked_image_urls:
            res = analyze_or_get_cached_url(u)
            if res.get("valid") and "emb" in res:
                liked_embs.append(res["emb"])
    if disliked_image_urls:
        for u in disliked_image_urls:
            res = analyze_or_get_cached_url(u)
            if res.get("valid") and "emb" in res:
                disliked_embs.append(res["emb"])

    scores = []
    notes = []
    audit_rows = []
    cache_hits = 0
    cache_misses = 0

    for idx, row in df_top.head(max_candidates).iterrows():
        urls = []
        if pd.notna(row.get('primary_photo')) and str(row.get('primary_photo')).startswith('http'):
            urls.append(str(row['primary_photo']))
        if pd.notna(row.get('alt_photos')):
            alt = [u.strip() for u in str(row['alt_photos']).split(',') if u.strip().startswith('http')]
            urls.extend(alt[:14])

        seen = set()
        urls = [u for u in urls if not (u in seen or seen.add(u))][:15]

        if not urls:
            scores.append(45.0)
            notes.append("No photo URLs (Unverified)")
            continue

        kitchen_candidates = [] # (k_prob, url, entry)
        living_candidates = []  # (l_prob, url, entry)

        for url in urls:
            if url in photo_cache and not force_refresh_vision:
                cache_hits += 1
            else:
                cache_misses += 1
            entry = analyze_or_get_cached_url(url)
            if not entry.get("valid"):
                continue

            if entry["ext_prob"] > 0.35 or entry["diag_prob"] > 0.35 or entry["b_prob"] > 0.40:
                continue
            if entry["k_prob"] >= 0.38 and entry.get("axes"):
                kitchen_candidates.append((entry["k_prob"], url, entry))
            elif entry["l_prob"] >= 0.38:
                living_candidates.append((entry["l_prob"], url, entry))

        kitchen_candidates.sort(key=lambda x: x[0], reverse=True)
        living_candidates.sort(key=lambda x: x[0], reverse=True)

        if not kitchen_candidates:
            floor_scores = [e["floor"] for _, _, e in living_candidates[:2]]
            fallback_floor = np.mean(floor_scores) if floor_scores else 45.0
            penalized = round(float(np.clip(fallback_floor * 0.65, 25.0, 58.0)), 1)
            scores.append(penalized)
            notes.append(f"No Kitchen Photo Found (Capped: {penalized}/100)")
            continue

        top_k = kitchen_candidates[:2]
        axis_results = {k: [] for k in kitchen_axes}
        floor_vals = []
        taste_vals = []

        for k_conf, k_url, k_entry in top_k:
            for axis_name in kitchen_axes:
                axis_results[axis_name].append(k_entry["axes"][axis_name])
            floor_vals.append(k_entry["floor"])

            if liked_embs or disliked_embs:
                emb = k_entry["emb"]
                pos_sim = max([float(np.dot(emb, le)) for le in liked_embs]) if liked_embs else 0.25
                neg_sim = max([float(np.dot(emb, de)) for de in disliked_embs]) if disliked_embs else 0.25
                margin = np.clip((pos_sim - neg_sim) * 350.0 + 50.0, 0.0, 100.0)
                taste_vals.append(float(margin))

        for _, _, l_entry in living_candidates[:2]:
            floor_vals.append(l_entry["floor"])

        app_s = float(np.mean(axis_results["appliances"]))
        cnt_s = float(np.mean(axis_results["counters"]))
        cab_s = float(np.mean(axis_results["cabinets"]))
        lay_s = float(np.mean(axis_results["layout"]))
        flr_s = float(np.mean(floor_vals)) if floor_vals else 50.0

        raw_weighted = (0.28 * app_s) + (0.26 * cnt_s) + (0.24 * cab_s) + (0.12 * lay_s) + (0.10 * flr_s)
        weakest_kitchen_link = min(app_s, cnt_s, cab_s)
        if weakest_kitchen_link < 35.0:
            raw_weighted = 0.50 * raw_weighted + 0.50 * weakest_kitchen_link

        if taste_vals:
            raw_weighted = 0.70 * raw_weighted + 0.30 * float(np.mean(taste_vals))

        final_v_score = round(float(np.clip(raw_weighted, 10.0, 100.0)), 1)
        scores.append(final_v_score)

        breakdown_str = (
            f"Kitchen {final_v_score}/100 "
            f"[App:{round(app_s)} | Cnt:{round(cnt_s)} | Cab:{round(cab_s)} | Open:{round(lay_s)} | Flr:{round(flr_s)}]"
        )
        notes.append(breakdown_str)

        audit_rows.append({
            "address": row.get("formatted_address", "Listing"),
            "price": row.get("list_price", ""),
            "url": row.get("property_url", "#"),
            "kitchen_img_url": top_k[0][1],
            "kitchen_conf": round(top_k[0][0] * 100),
            "final_score": final_v_score,
            "appliances": round(app_s),
            "counters": round(cnt_s),
            "cabinets": round(cab_s),
            "layout": round(lay_s),
            "flooring": round(flr_s)
        })

    if cache_dirty[0]:
        with open(cache_path, "wb") as f:
            pickle.dump(photo_cache, f)
        print(f"✅ Saved {len(photo_cache)} analyzed photo embeddings to {cache_path} (Hits: {cache_hits}, New: {cache_misses})")
    else:
        print(f"⚡ 100% CLIP Cache Hit ({cache_hits} photos)! Evaluated all kitchens & personal taste vectors in <0.1s!")

    if export_audit_html and audit_rows:
        os.makedirs(os.path.dirname(export_audit_html), exist_ok=True)
        cards_html = ""
        for r in sorted(audit_rows, key=lambda x: x["final_score"], reverse=True):
            badge_color = "#1b7e3c" if r["final_score"] >= 75 else ("#b57c00" if r["final_score"] >= 55 else "#b82e2e")
            cards_html += f"""
            <div style="border:1px solid #ddd; border-radius:8px; width:340px; overflow:hidden; background:#fff; box-shadow:0 2px 5px rgba(0,0,0,0.08);">
                <img src="{r['kitchen_img_url']}" style="width:100%; height:210px; object-fit:cover;" />
                <div style="padding:12px; font-family:sans-serif; font-size:13px;">
                    <div style="display:flex; justify-content:space-between; align-items:center;">
                        <span style="background:{badge_color}; color:#fff; padding:3px 8px; border-radius:4px; font-weight:bold;">
                            Kitchen Score: {r['final_score']}/100
                        </span>
                        <b>${r['price']}/mo</b>
                    </div>
                    <div style="margin:6px 0; font-weight:bold;">{r['address']}</div>
                    <div style="color:#444; line-height:1.5;">
                        • <b>Appliances (Stainless vs Coil):</b> {r['appliances']}/100<br>
                        • <b>Counters (Stone vs Laminate):</b> {r['counters']}/100<br>
                        • <b>Cabinets (Modern vs Dated Oak):</b> {r['cabinets']}/100<br>
                        • <b>Open Island Layout:</b> {r['layout']}/100<br>
                        • <b>Hard Flooring (No Carpet):</b> {r['flooring']}/100
                    </div>
                    <div style="margin-top:8px;">
                        <a href="{r['url']}" target="_blank" style="color:#0066cc; text-decoration:none; font-weight:bold;">Open Listing →</a>
                    </div>
                </div>
            </div>
            """
        html_doc = f"""<!DOCTYPE html>
        <html><head><meta charset="utf-8"><title>Kitchen Visual Audit Report</title></head>
        <body style="background:#f5f6f8; padding:24px; font-family:sans-serif;">
            <h2>🍳 Multi-Axis Kitchen & Interior Visual Audit Report</h2>
            <p>Showing the exact photo classified as the <b>Kitchen</b> for each candidate and its 5-axis sub-scores (weakest-link penalized):</p>
            <div style="display:flex; flex-wrap:wrap; gap:16px;">{cards_html}</div>
        </body></html>"""
        with open(export_audit_html, "w") as f:
            f.write(html_doc)
        print(f"Saved Visual Kitchen Audit card grid to {export_audit_html}!")

    df_top = df_top.copy()
    df_top.loc[df_top.head(max_candidates).index, 'vision_modern_score'] = scores
    df_top.loc[df_top.head(max_candidates).index, 'vision_notes'] = notes
    return df_top


PREFS = load_preferences(PREF_FILE)
liked_urls = PREFS['taste_profile'].get('liked_kitchen_photo_urls', [])
disliked_urls = PREFS['taste_profile'].get('disliked_kitchen_photo_urls', [])
CLIP_CACHE_FILE = os.path.join(CACHE_DIR, 'clip_photo_cache.pkl')
FORCE_REFRESH_VISION = False  # Set True only if you change the text prompts in kitchen_axes

print(f'Calibrating CLIP Vision with {len(liked_urls)} Liked and {len(disliked_urls)} Disliked kitchen photos from memory...')

scored_df = score_photos_with_local_clip(
    scored_df,
    max_candidates=25,
    liked_image_urls=liked_urls,
    disliked_image_urls=disliked_urls,
    export_audit_html='outputs/visual_kitchen_audit.html',
    cache_path=CLIP_CACHE_FILE,
    force_refresh_vision=FORCE_REFRESH_VISION
)

# Blend NLP text score (25%) + Strict 5-Axis Kitchen Vision score (75%)
has_v = scored_df['vision_modern_score'].notna()
scored_df.loc[has_v, 'modern_interior_score'] = (
    0.25 * scored_df.loc[has_v, 'modern_interior_score'] +
    0.75 * scored_df.loc[has_v, 'vision_modern_score']
).round(1)

weights = CONFIG['weights']
scored_df['composite_score'] = sum(scored_df.get(k, 50.0) * w for k, w in weights.items()).round(1)
scored_df = scored_df.sort_values(by='composite_score', ascending=False).reset_index(drop=True)

HISTORY_FILE = os.path.join(CACHE_DIR, 'seen_listings_history.json')
scored_df = track_multi_week_changes_and_favorites(scored_df, PREFS, history_path=HISTORY_FILE)
os.makedirs('outputs', exist_ok=True)
scored_df.to_csv('outputs/san_antonio_ranked_rentals_v4.csv', index=False)
print('✅ Saved outputs/san_antonio_ranked_rentals_v4.csv and outputs/visual_kitchen_audit.html!')
scored_df[['listing_status', 'composite_score', 'formatted_address', 'list_price', 'property_age_years', 'modern_interior_score', 'vision_notes', 'swri_rush_hour_mins', 'nearest_grocery']].head(15)


In [ ]:
# @title Step 6: Manage Long-Term Favorites, Blacklist & Kitchen Taste Anchors
# Run `record_feedback()` below whenever you want to:
#  1. Add a property to your ⭐ Favorites Shortlist (`liked=True`) -> exports to `outputs/favorites_shortlist.csv`
#  2. Permanently 🚫 Blacklist an address (`liked=False`) or an entire apartment complex (`blacklist_complex_name='...'`)
#  3. Add a Liked or Disliked Kitchen Photo URL (`kitchen_photo_url='https://...'`) to calibrate CLIP Vision

# --- EXAMPLES (Uncomment & run any time) ---
# record_feedback(
#     address="123 Example St, San Antonio, TX 78238",
#     liked=True,
#     kitchen_photo_url="https://ap.rdcpix.com/...jpg",
#     reason="Love the quartz island, gas range, and attached garage!",
#     path=PREF_FILE
# )

# record_feedback(
#     address="456 Dated Apt, San Antonio, TX",
#     liked=False,
#     blacklist_complex_name=" Villas at Example ",  # Blocks ALL units from this complex!
#     kitchen_photo_url="https://ap.rdcpix.com/...jpg",
#     reason="Old 1990s oak cabinets and coil stove",
#     path=PREF_FILE
# )
print("Memory Manager ready! Current Favorites:", PREFS['taste_profile']['liked_property_addresses'])


In [ ]:
# @title Step 7: Single-Link / Address Sanity Check & Score Explainer (`inspect_and_explain_listing`)
import os
import re
import numpy as np
import pandas as pd
import geopandas as gpd
from homeharvest import scrape_property

def address_from_url_or_query(url_or_address):
    """Extracts a searchable street address/zip from a Realtor.com, Zillow, or Redfin URL, or returns the raw address string."""
    s = str(url_or_address).strip()
    if not s.startswith("http"):
        return s
    # Realtor.com URL pattern: .../realestateandhomes-detail/123-Main-St_San-Antonio_TX_78238_M12345...
    m_realtor = re.search(r"realestateandhomes-detail/([^/?#]+)", s)
    if m_realtor:
        slug = m_realtor.group(1)
        slug = re.sub(r"_M[0-9_-]+$", "", slug)
        parts = slug.split("_")
        return " ".join(p.replace("-", " ") for p in parts)
    # Zillow URL pattern: .../homedetails/123-Main-St-San-Antonio-TX-78238/12345_zpid/
    m_zillow = re.search(r"homedetails/([^/?#]+)", s)
    if m_zillow:
        return m_zillow.group(1).replace("-", " ")
    return s

def inspect_and_explain_listing(
    url_or_address,
    raw_df,
    ctx,
    active_criteria,
    vision_fn=None,
    liked_urls=None,
    disliked_urls=None,
    clip_cache_path="cache/clip_photo_cache.pkl",
    add_to_favorites=False,
    add_to_blacklist=False,
    pref_file="data/preferences.json",
    record_feedback_fn=None
):
    """
    Single-Link / Address Sanity Check & Diagnostic Waterfall Inspector:
    1. Finds the property in `raw_df` (by URL or fuzzy address match) or scrapes it on-the-fly via HomeHarvest.
    2. Checks every Hard Cutoff Gate (Budget, Beds/Baths, In-Unit Laundry/Dishwasher, Max Commute, Max Grocery/Park Dist)
       and shows PASS / FAIL with the exact value vs threshold.
    3. Runs all 12 criteria + 5-axis CLIP Kitchen Vision and outputs a full Waterfall Table:
       [Criterion | Raw Score (0-100) | Weight | Weighted Pts | Fleet Percentile | Diagnostic Note]
    4. Computes "How to Tweak Weights to Get More Like This": identifies which criteria this property excels at
       relative to the San Antonio average and suggests exact `CONFIG['weights']` adjustments!
    """
    query_str = str(url_or_address).strip()
    parsed_addr = address_from_url_or_query(query_str)
    print(f"🔍 Inspecting target listing: '{parsed_addr}'...")

    matches = pd.DataFrame()
    if raw_df is not None and len(raw_df) > 0:
        if query_str.startswith("http") and "property_url" in raw_df.columns:
            matches = raw_df[raw_df["property_url"].astype(str).str.contains(query_str.split("?")[0], case=False, na=False)]
        if len(matches) == 0 and "formatted_address" in raw_df.columns:
            # Try matching street number + street name tokens
            tokens = [t for t in re.split(r"\W+", parsed_addr.lower()) if t and t not in ("san", "antonio", "tx", "apt", "unit")]
            if tokens:
                mask = pd.Series(True, index=raw_df.index)
                for tok in tokens[:3]:
                    mask = mask & raw_df["formatted_address"].astype(str).str.lower().str.contains(tok, na=False)
                matches = raw_df[mask]

    if len(matches) == 0:
        print(f"🌐 Not in local scrape cache — fetching live via HomeHarvest for '{parsed_addr}'...")
        try:
            live_df = scrape_property(location=parsed_addr, listing_type="for_rent", past_days=90)
            if live_df is not None and len(live_df) > 0:
                matches = live_df.dropna(subset=["latitude", "longitude"]).head(1).copy()
        except Exception as e:
            print(f"⚠️ Live scrape error: {e}")

    if len(matches) == 0:
        print("❌ Could not locate listing in cache or live scrape. Try passing the street address (e.g. '1234 Culebra Rd, San Antonio, TX').")
        return None

    target_df = matches.head(1).copy().reset_index(drop=True)
    gdf = gpd.GeoDataFrame(target_df, geometry=gpd.points_from_xy(target_df.longitude, target_df.latitude), crs="EPSG:4326").to_crs(epsg=32614)
    target_df["utm_x"], target_df["utm_y"] = gdf.geometry.x, gdf.geometry.y

    row = target_df.iloc[0]
    m = {}
    for fn in active_criteria:
        m.update(fn(row, ctx))

    single_scored = pd.concat([target_df, pd.DataFrame([m])], axis=1)

    if vision_fn is not None:
        single_scored = vision_fn(
            single_scored,
            max_candidates=1,
            liked_image_urls=liked_urls,
            disliked_image_urls=disliked_urls,
            export_audit_html=None,
            cache_path=clip_cache_path
        )
        if pd.notna(single_scored.loc[0, "vision_modern_score"]):
            single_scored.loc[0, "modern_interior_score"] = round(
                0.25 * float(single_scored.loc[0, "modern_interior_score"]) +
                0.75 * float(single_scored.loc[0, "vision_modern_score"]), 1
            )

    r = single_scored.iloc[0]
    weights = ctx["config"]["weights"]
    comp_score = round(sum(float(r.get(k, 50.0)) * w for k, w in weights.items()), 1)
    single_scored.loc[0, "composite_score"] = comp_score

    # 1. Print Header & Hard Gatekeeper Audit
    addr = r.get("formatted_address", parsed_addr)
    price = r.get("list_price", "N/A")
    beds = r.get("beds", "N/A")
    baths = r.get("full_baths", "N/A")
    f_cfg = ctx["config"]["filters"]
    c_cfg = ctx["config"]["cutoffs"]

    print("=" * 88)
    print(f"🏠 LISTING SANITY CHECK: {addr}")
    print(f"💵 Price: ${price}/mo | 🛏️ {beds} Beds / {baths} Baths | 🏗️ Age: {r.get('property_age_years', 'Unknown')} yrs | 🏆 Composite Score: {comp_score}/100")
    print("-" * 88)
    print("🚦 HARD CUTOFF GATEKEEPER AUDIT (Why was this kept or filtered out?):")
    gates = [
        ("Rent Budget", f"${price}", f"${f_cfg['min_rent']} - ${f_cfg['max_rent']}", f_cfg["min_rent"] <= float(price or 0) <= f_cfg["max_rent"]),
        ("Min Bedrooms", f"{beds} beds", f">= {f_cfg['min_beds']} beds", float(beds or 0) >= f_cfg["min_beds"]),
        ("Appliances Gate", r.get("appliances_status", ""), "In-Unit W/D + Dishwasher", bool(r.get("passes_appliance_gate", True))),
        ("SwRI Rush-Hour Commute", f"{r.get('swri_rush_hour_mins')} mins", f"<= {c_cfg['max_swri_commute_mins']} mins", float(r.get("swri_rush_hour_mins", 99)) <= c_cfg["max_swri_commute_mins"]),
        ("Grocery Distance (No Walmart)", f"{r.get('grocery_dist_m')} m ({r.get('nearest_grocery')})", f"<= {c_cfg['max_grocery_dist_m']} m", float(r.get("grocery_dist_m", 9999)) <= c_cfg["max_grocery_dist_m"]),
        ("Park/Greenway Distance", f"{r.get('park_dist_m')} m ({r.get('nearest_park_trail')})", f"<= {c_cfg['max_park_trail_dist_m']} m", float(r.get("park_dist_m", 9999)) <= c_cfg["max_park_trail_dist_m"])
    ]
    for g_name, actual, thresh, passed in gates:
        status = "✅ PASS" if passed else "❌ FAIL (FILTERED OUT BY THIS CUTOFF!)"
        print(f"  • {g_name:<30}: {actual:<32} (Rule: {thresh}) -> {status}")

    # 2. Build Criterion-by-Criterion Waterfall Table
    note_map = {
        "commute_score": f"{r.get('swri_rush_hour_mins')} mins ({r.get('commute_corridor')})",
        "grocery_score": f"{r.get('grocery_walk_mins')}m walk / {r.get('grocery_bike_mins')}m bike to {r.get('nearest_grocery')}",
        "park_trail_score": f"{r.get('park_walk_mins')}m walk to {r.get('nearest_park_trail')}",
        "modern_interior_score": f"Vision: {r.get('vision_notes', 'N/A')} | Text: {r.get('interior_tags', '')}",
        "pollution_highway_score": f"{r.get('highway_buffer_m')}m from nearest highway",
        "garage_bike_score": f"{r.get('garage_bike_features')}",
        "costco_score": f"{r.get('costco_drive_mins')}m drive to {r.get('nearest_costco')}",
        "recency_score": f"Age: {r.get('property_age_years')} yrs (Built {r.get('year_built', 'N/A')})",
        "water_softener_score": f"{r.get('water_softener_tags')}",
        "pet_flooring_score": f"{r.get('pet_flooring_tags')}",
        "effective_rent_score": f"Extracted Fees: ${r.get('extracted_monthly_fees', 0)}/mo | Utils Incl: {r.get('utilities_included')}",
        "noise_buffer_score": f"Rail: {r.get('railway_dist_m')}m | SAT: {r.get('sat_airport_dist_km')}km ({r.get('noise_buffer_notes')})"
    }

    rows_table = []
    for k, w in sorted(weights.items(), key=lambda x: x[1], reverse=True):
        val = float(r.get(k, 50.0))
        pts = round(val * w, 2)
        rows_table.append({
            "Criterion": k,
            "Raw Score (0-100)": val,
            "Weight": w,
            "Weighted Pts": pts,
            "Details / Why It Scored This": note_map.get(k, "")
        })
    breakdown_df = pd.DataFrame(rows_table)

    # 3. Suggest Weight Tweaks if the user LOVES this listing and wants more like it
    print("-" * 88)
    print("🎛️ HOW TO TWEAK SCORING TO GET MORE PLACES LIKE THIS:")
    strengths = [row_d["Criterion"] for row_d in rows_table if row_d["Raw Score (0-100)"] >= 80.0]
    bottlenecks = [row_d["Criterion"] for row_d in rows_table if row_d["Raw Score (0-100)"] < 55.0]
    failed_gates = [g[0] for g in gates if not g[3]]
    if failed_gates:
        print(f"  ⚠️ First: This listing was blocked by hard cutoffs: {failed_gates}. Relax those in `CONFIG['cutoffs']` if you want it included!")
    if strengths:
        print(f"  📈 Signature Strengths (>=80/100): {strengths} -> Boost these weights in `CONFIG['weights']` to surface similar properties.")
    if bottlenecks:
        print(f"  📉 Dragging Score Down (<55/100): {bottlenecks} -> Lower these weights if you don't care about them for this style of home.")
    print("=" * 88)

    if record_feedback_fn is not None and (add_to_favorites or add_to_blacklist):
        record_feedback_fn(
            address=addr,
            liked=bool(add_to_favorites and not add_to_blacklist),
            reason="Added via Single-Link Inspector",
            path=pref_file
        )

    return breakdown_df


# Paste ANY Realtor.com, Zillow, or Redfin URL — or any San Antonio street address — below!
# It will look it up in your cached scrape (or fetch it live), test every Hard Gatekeeper Cutoff,
# print a full 12-criterion Waterfall Breakdown, and tell you how to tweak `CONFIG['weights']` to get more places like it!

TARGET_URL_OR_ADDRESS = ""  # <-- Paste a link or address here (e.g., scored_df.iloc[0]['formatted_address'] or a Realtor.com URL)

if TARGET_URL_OR_ADDRESS.strip():
    waterfall_df = inspect_and_explain_listing(
        TARGET_URL_OR_ADDRESS,
        raw_df=raw if 'raw' in locals() else None,
        ctx={'config': CONFIG, 'trees': TREES},
        active_criteria=ACTIVE_CRITERIA,
        vision_fn=score_photos_with_local_clip,
        liked_urls=PREFS['taste_profile'].get('liked_kitchen_photo_urls', []),
        disliked_urls=PREFS['taste_profile'].get('disliked_kitchen_photo_urls', []),
        clip_cache_path=os.path.join(CACHE_DIR, 'clip_photo_cache.pkl'),
        add_to_favorites=False,   # Flip to True to automatically save this link to your Favorites Shortlist!
        add_to_blacklist=False,   # Flip to True to permanently blacklist this property!
        pref_file=PREF_FILE,
        record_feedback_fn=record_feedback
    )
    display(waterfall_df)
else:
    print("💡 Paste any listing URL or street address into `TARGET_URL_OR_ADDRESS` above and run this cell to audit it!")
